In [0]:
%run /Workspace/Repos/projects/databricks-batch-pipeline/modules/silver_layer

In [0]:
data = {
    "table_name": "hr",
    "schema_details": {
        "Employee_name": "string",
        "EmpID": "string",
        "Salary": "string",
        "HispanicLatino": "string",
        "EngagementSurvey": "double",
        "EmpSatisfaction": "int",
    },
    "keys": ["EmpID"],
    "write_mode": "scd2"        
}

In [0]:
pipeline_name = "hr"
conf = (
    spark.table("session_11_framework.config_table")
    .filter(col("pipeline_name") == pipeline_name)
    .select(
        "table_name",
        "schema_details",
        "keys",
        "write_mode"
        )
    ).first()
conf

In [0]:
silver_pipeline = SilverLayer(
    table_name = conf.table_name,
    schema_details = conf.schema_details,
    keys = conf.keys,
    write_mode = conf.write_mode
)
df = silver_pipeline.read_data()
invalid_df = silver_pipeline.get_invalid_record(df)
key_null_df = silver_pipeline.get_key_null_record(df)
dup_df = silver_pipeline.get_dup_record(df, key_null_df)
all_bad_df = silver_pipeline.get_all_bad_records(
    invalid_df = invalid_df, 
    key_null_df = key_null_df, 
    duplicate_df = dup_df
)
cleaned_df = silver_pipeline.get_cleansing_df(df, all_bad_df)
load_bad_records = silver_pipeline.load_bad_record(all_bad_df)
load_to_silver_layer = silver_pipeline.load_to_silver_layer(cleaned_df)


In [0]:
df = spark.table("hr_silver").display()
